In [227]:
import pandas as pd

data = pd.read_csv("https://confrecordings.ams3.digitaloceanspaces.com/reviews_amazon.csv")

print(data.shape)
print(data.head())

(100, 2)
                 Summary                                               Text
0  Good Quality Dog Food  I have bought several of the Vitality canned d...
1      Not as Advertised  Product arrived labeled as Jumbo Salted Peanut...
2  "Delight" says it all  This is a confection that has been around a fe...
3         Cough Medicine  If you are looking for the secret ingredient i...
4            Great taffy  Great taffy at a great price.  There was a wid...


In [228]:
import re

def textcleaner(text, num):
    newString = text.lower()
    newString = re.sub(r'[^a-z\s]', ' ', newString)
    newString = re.sub(r'\s+', ' ', newString).strip()

    # Only remove the most basic structural words
    word_stop = {"the", "a", "an"}

    tokens = newString.split()
    if num == 0:
        tokens = [word for word in tokens if word not in word_stop]

    return " ".join(tokens)

In [229]:
import numpy as np

data['Text'] = data['Text'].fillna('')
data['Summary'] = data['Summary'].fillna('')

data['Cleaned Text'] = data['Text'].apply(lambda x: textcleaner(x, 0))
data['Cleaned Summary'] = data['Summary'].apply(lambda x: textcleaner(x, 0))

data.replace('', np.nan, inplace=True)
data.dropna(axis=0, inplace=True)

print(data.head(2))

                 Summary                                               Text  \
0  Good Quality Dog Food  I have bought several of the Vitality canned d...   
1      Not as Advertised  Product arrived labeled as Jumbo Salted Peanut...   

                                        Cleaned Text        Cleaned Summary  
0  i have bought several of vitality canned dog f...  good quality dog food  
1  product arrived labeled as jumbo salted peanut...      not as advertised  


In [230]:
cleaned_text = np.array(data['Cleaned Text'])
cleaned_summary = np.array(data['Cleaned Summary'])

short_text = []
short_summary = []

for i in range(len(cleaned_text)):
    short_text.append(cleaned_text[i])
    short_summary.append(cleaned_summary[i])

df = pd.DataFrame()
df['Text'] = short_text
df['Summary'] = ['start ' + s + ' end' for s in short_summary]

print(df.head(2))

                                                Text  \
0  i have bought several of vitality canned dog f...   
1  product arrived labeled as jumbo salted peanut...   

                           Summary  
0  start good quality dog food end  
1      start not as advertised end  


In [231]:
from sklearn.model_selection import train_test_split

X = np.array(df['Text'])
Y = np.array(df['Summary'])

x_tr, x_val, y_tr, y_val = train_test_split(X, Y, test_size=0.1, random_state=0, shuffle=True)

print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(90,)
(10,)
(90,)
(10,)


In [232]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_text_len = 250

x_tokenizer = Tokenizer()
x_tokenizer.fit_on_texts(list(x_tr))

thresh = 1
cnt, tot_cnt, freq, tot_freq = 0, 0, 0, 0
for key, value in x_tokenizer.word_counts.items():
    tot_cnt += 1
    tot_freq += value
    if value < thresh:
        cnt += 1
        freq += value

print("% of rare words in vocabulary:", round((cnt / tot_cnt) * 100, 2))
print("Total Coverage of rare words:", round((freq / tot_freq) * 100, 2))

x_tokenizer = Tokenizer(num_words=tot_cnt - cnt)
x_tokenizer.fit_on_texts(list(x_tr))

x_tr_seq = x_tokenizer.texts_to_sequences(x_tr)
x_val_seq = x_tokenizer.texts_to_sequences(x_val)

x_tr = pad_sequences(x_tr_seq, maxlen=max_text_len, padding='post')
x_val = pad_sequences(x_val_seq, maxlen=max_text_len, padding='post')

x_voc = x_tokenizer.num_words + 1

y_tokenizer = Tokenizer()
y_tokenizer.fit_on_texts(list(y_tr))

thresh = 1
cnt, tot_cnt, freq, tot_freq = 0, 0, 0, 0
for key, value in y_tokenizer.word_counts.items():
    tot_cnt += 1
    tot_freq += value
    if value < thresh:
        cnt += 1
        freq += value

print("% of rare words in summary vocab:", round((cnt / tot_cnt) * 100, 2))
print("Total Coverage of rare words in summary:", round((freq / tot_freq) * 100, 2))

y_tokenizer = Tokenizer(num_words=tot_cnt - cnt)
y_tokenizer.fit_on_texts(list(y_tr))

y_tr_seq = y_tokenizer.texts_to_sequences(y_tr)
y_val_seq = y_tokenizer.texts_to_sequences(y_val)

y_tr = pad_sequences(y_tr_seq, maxlen=max_summary_len, padding='post')
y_val = pad_sequences(y_val_seq, maxlen=max_summary_len, padding='post')

y_voc = y_tokenizer.num_words + 1
print("Final y vocabulary size:", y_voc)

% of rare words in vocabulary: 0.0
Total Coverage of rare words: 0.0
% of rare words in summary vocab: 0.0
Total Coverage of rare words in summary: 0.0
Final y vocabulary size: 196


In [233]:
ind=[]
for i in range(len(y_tr)):
    cnt=0
    for j in y_tr[i]:
        if j!=0:
            cnt=cnt+1
    if(cnt==2):
        ind.append(i)
y_tr=np.delete(y_tr,ind, axis=0)
x_tr=np.delete(x_tr,ind, axis=0)

In [234]:
ind=[]
for i in range(len(y_val)):
    cnt=0
    for j in y_val[i]:
        if j!=0:
            cnt=cnt+1
    if(cnt==2):
        ind.append(i)
y_val=np.delete(y_val,ind, axis=0)
x_val=np.delete(x_val,ind, axis=0)

In [235]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate, TimeDistributed, Attention
from tensorflow.keras.models import Model

latent_dim = 32
embedding_dim = 32

encoder_inputs = Input(shape=(max_text_len,))
enc_emb = Embedding(x_voc, embedding_dim, trainable=True)(encoder_inputs)

encoder_lstm1 = LSTM(latent_dim, return_sequences=True, return_state=True, dropout=0.4, recurrent_dropout=0.4)
encoder_output1, state_h1, state_c1 = encoder_lstm1(enc_emb)

decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(y_voc, embedding_dim, trainable=True)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, dropout=0.4, recurrent_dropout=0.2)
decoder_outputs, decoder_fwd_state, decoder_back_state = decoder_lstm(dec_emb, initial_state=[state_h1, state_c1])

attention = Attention()
attn_out = attention([decoder_outputs, encoder_output1])

merge = Concatenate(axis=-1, name='concat_layer1')([decoder_outputs, attn_out])

decoder_dense = TimeDistributed(Dense(y_voc, activation='softmax'))
decoder_outputs = decoder_dense(merge)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

print(model.summary())

Model: "functional_40"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_67      │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_68      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_28        │ (None, 250, 32)   │     43,232 │ input_layer_67[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_29        │ (None, None, 32)  │      6,272 │ input_layer_68[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_28 (LSTM)      │ [(None, 250, 32), │      8,320 │ embedding_28[0][… │
│                     │ (None, 32),       │            │                   │
│                     │ (None, 32)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_29 (LSTM)      │ [(None, None,     │      8,320 │ embedding_29[0][… │
│                     │ 32), (None, 32),  │            │ lstm_28[0][1],    │
│                     │ (None, 32)]       │            │ lstm_28[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_27        │ (None, None, 32)  │          0 │ lstm_29[0][0],    │
│ (Attention)         │                   │            │ lstm_28[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_layer1       │ (None, None, 64)  │          0 │ lstm_29[0][0],    │
│ (Concatenate)       │                   │            │ attention_27[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_14 │ (None, None, 196) │     12,740 │ concat_layer1[0]… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 78,884 (308.14 KB)

 Trainable params: 78,884 (308.14 KB)

 Non-trainable params: 0 (0.00 B)

None


In [236]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy')
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

# Training the model
# Note: y_tr[:, :-1] is the input to decoder, y_tr[:, 1:] is the target output
history = model.fit(
    [x_tr, y_tr[:, :-1]],
    y_tr.reshape(y_tr.shape[0], y_tr.shape[1], 1)[:, 1:],
    epochs=300,
    batch_size=16,
    callbacks=[es],
    validation_data=([x_val, y_val[:, :-1]], y_val.reshape(y_val.shape[0], y_val.shape[1], 1)[:, 1:])
)

encoder_model = Model(inputs=encoder_inputs, outputs=[encoder_output1,state_h1, state_c1])

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_hidden_state_input = Input(shape=(max_text_len, latent_dim))

dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])

attention_layer = Attention()
attn_out2 = attention_layer([decoder_outputs2, decoder_hidden_state_input])

merge2 = Concatenate(axis=-1)([decoder_outputs2, attn_out2])

decoder_outputs2 = decoder_dense(merge2)

decoder_model = Model(
    [decoder_inputs, decoder_hidden_state_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2, state_h2, state_c2]
)


Epoch 1/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 14s 604ms/step - loss: 5.2153 - val_loss: 5.0622
Epoch 2/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 230ms/step - loss: 4.9777 - val_loss: 4.7085
Epoch 3/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 231ms/step - loss: 4.6236 - val_loss: 4.1524
Epoch 4/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 228ms/step - loss: 4.0964 - val_loss: 3.4032
Epoch 5/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 236ms/step - loss: 3.4466 - val_loss: 2.6313
Epoch 6/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 256ms/step - loss: 2.7591 - val_loss: 1.9820
Epoch 7/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 291ms/step - loss: 2.2524 - val_loss: 1.5878
Epoch 8/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 245ms/step - loss: 2.0066 - val_loss: 1.4223
Epoch 9/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 236ms/step - loss: 1.9336 - val_loss: 1.3489
Epoch 10/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 227ms/step - loss: 1.8987 - val_loss: 1.3049
Epoch 11/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 232ms/step - loss: 1.8649 - val_loss: 1.2796
Epoch 12/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 226ms/step - loss: 1.

In [237]:
reverse_target_word_index = y_tokenizer.index_word
reverse_source_word_index = x_tokenizer.index_word
target_word_index = y_tokenizer.word_index
reverse_target_word_index[0] = '<PAD>'

In [238]:
def decode_sequence(input_seq):
    # 1. Encode the input as state vectors.
    # Note: en_out is required if your model uses an Attention layer.
    en_out, en_h, en_c = encoder_model.predict(input_seq, verbose=0)

    # 2. Generate empty target sequence of length 1 with only the start token.
    target_seq = np.zeros((1, 1))
    start_index = target_word_index.get('start')

    if start_index is None:
        print("Error: 'start' token not found in target_word_index")
        return ""

    target_seq[0, 0] = start_index

    stop_condition = False
    decoded_sentence = ""

    # 3. Initialize internal states with encoder states
    states_value = [en_h, en_c]

    while not stop_condition:
        # Pass current target token, the fixed encoder output, and rolling states
        # The input list order [target_seq, en_out] + states_value must match your
        # decoder_model's input definition exactly.
        output_tokens, h, c = decoder_model.predict(
            [target_seq, en_out] + states_value,
            verbose=0
        )

        # 4. Sample the word with highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_word_index.get(sampled_token_index, '')

        # 5. Check for exit conditions
        current_word_count = len(decoded_sentence.split())

        if (sampled_char == 'end' or current_word_count >= 4):
            stop_condition = True
        else:
            if sampled_char != 'start' and sampled_char != '':
                decoded_sentence += sampled_char + " "

        # 6. Update the target sequence (the predicted word is the next input)
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # 7. Update internal states for the next iteration
        states_value = [h, c]

    return decoded_sentence.strip()

In [239]:
def seq2summary(input_seq):
    words = []
    for i in input_seq:
        if i != 0 and i != target_word_index.get('start', None) and i != target_word_index.get('end', None):
            words.append(reverse_target_word_index.get(i, ''))
    return ' '.join(words)

def seq2text(input_seq):
    words = []
    for i in input_seq:
        if i != 0:
            words.append(reverse_source_word_index.get(i, ''))
    return ' '.join(words)

for i in range(5):
    print('Review:', seq2text(x_tr[i]))
    print('Original summary:', seq2summary(y_tr[i]))
    print('Predicted summary:', decode_sequence(x_tr[i].reshape(1, max_text_len)))
    print('\n')

Review: we re used to spicy foods down here in south texas and these are not at all spicy doubt very much habanero is used at all could take it up notch or two
Original summary: not ass kickin
Predicted summary: great


Review: this food is great all ages dogs i have year old and puppy they are both so soft and hardly ever get sick food is good especially when you have amazon prime shipping
Original summary: mmmmm mmmmm good
Predicted summary: great


Review: taste was great but berries had melted may order again in winter if you order in cold weather you should enjoy flavor
Original summary: order only in cold weather
Predicted summary: great


Review: good flavor these came securely packed they were fresh and delicious i love these twizzlers
Original summary: fresh and greasy
Predicted summary: great


Review: this taffy is so good it is very soft and chewy flavors are amazing i would definitely recommend you buying it very satisfying
Original summary: wonderful tasty taffy
Predicted